In [ ]:
import os, copy, json, time, random, shutil
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from IPython.display import display

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

WINDOW = 7
HORIZONS = [1, 3, 7]
HMAX = max(HORIZONS)
MAX_FILL = 3
TRAIN_FRAC, VAL_FRAC = 0.70, 0.15

POLLUTANTS = ["PM2.5", "PM10", "NO", "NO2", "NOx", "CO", "SO2", "O3"]
DROP_CITIES = ["Mumbai", "Jorapokhar"]
DROP_COLS = ["AQI", "AQI_Bucket", "Xylene", "Toluene", "Benzene", "NH3"]
NORTH_CITIES = ["Delhi", "Lucknow", "Patna", "Gurugram", "Amritsar", "Chandigarh", "Jaipur"]

PARAM_CAP = 60_000
N_TRIALS = 6
SEARCH_SEED = 0
FINAL_SEEDS = [0, 1, 2]
MAX_EPOCHS = 100
PATIENCE = 12
INCLUDE_REFERENCE = True
RUN_SIZE_SENSITIVITY = False

BASELINE = "MLP"
MODEL_NAMES = ["MLP", "1D-CNN", "LSTM/GRU", "Transformer"]
BAND_EDGES = [30, 60, 90, 120, 250]
BAND_LABELS = ["Good", "Satisfactory", "Moderate", "Poor", "Very Poor", "Severe"]
SPIKE_THRESHOLD = 150
LOCKDOWN_START = "2020-03-25"
PLOT_CITIES = ["Delhi", "Chennai"]
PLOT_HORIZON = 3
OUT_DIR = "dlproject_outputs"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
AUTOCAST_DEVICE = "cuda" if DEVICE == "cuda" else "cpu"
USE_AMP = DEVICE == "cuda"

for sub in ["data/raw", "data/clean", "results", "figures"]:
    os.makedirs(os.path.join(OUT_DIR, sub), exist_ok=True)

COLORS = {"MLP": "#1f77b4", "1D-CNN": "#2ca02c", "LSTM/GRU": "#ff7f0e", "Transformer": "#d62728",
          "Persistence": "#7f7f7f", "7-day average": "#bcbd22", "Actual": "#000000"}

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.titlesize": 11, "axes.labelsize": 10,
                     "legend.fontsize": 9, "legend.frameon": False})

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

def finish(fig, name):
    fig.tight_layout()
    fig.savefig(os.path.join(OUT_DIR, "figures", f"{name}.png"), dpi=200, bbox_inches="tight")
    plt.show()
    plt.close(fig)

set_seed(42)
print(f"PyTorch {torch.__version__} | Device: {DEVICE}" + (f" ({torch.cuda.get_device_name(0)})" if DEVICE == "cuda" else ""))
if DEVICE == "cpu":
    print("A GPU runtime is not active. Select Runtime > Change runtime type > GPU for faster training.")

In [ ]:
def locate_dataset():
    found = [f for f in sorted(os.listdir(".")) if f.lower().endswith(".csv") and "city_day" in f.lower()]
    if found:
        print(f"Dataset found in the working directory: {found[0]}")
        return found[0]
    if IN_COLAB:
        print("Select the dataset file (city_day.csv) in the dialog below.")
        uploaded = files.upload()
        names = [n for n in uploaded if n.lower().endswith(".csv")]
        if not names:
            raise FileNotFoundError("No CSV file was uploaded.")
        return names[0]
    raise FileNotFoundError("Place city_day.csv in the working directory and run the cell again.")

DATA_FILE = locate_dataset()
raw = pd.read_csv(DATA_FILE, parse_dates=["Date"])

missing_required = {"City", "Date", *POLLUTANTS} - set(raw.columns)
if missing_required:
    raise ValueError(f"The dataset is missing required columns: {sorted(missing_required)}")

shutil.copyfile(DATA_FILE, os.path.join(OUT_DIR, "data", "raw", "city_day_raw.csv"))
print(f"Loaded {len(raw):,} records and {raw.shape[1]} columns from {DATA_FILE}")

In [ ]:
df = (raw[~raw["City"].isin(DROP_CITIES)]
      .drop(columns=[c for c in DROP_COLS if c in raw.columns])
      .sort_values(["City", "Date"])
      .reset_index(drop=True))

frames = []
for city, g in df.groupby("City"):
    idx = pd.date_range(g["Date"].min(), g["Date"].max(), freq="D")
    g = g.set_index("Date")[POLLUTANTS].reindex(idx)
    g.index.name = "Date"
    g["City"] = city
    frames.append(g.reset_index())
daily = pd.concat(frames, ignore_index=True)
daily_raw = daily.copy()

both = daily["PM2.5"].notna() & daily["PM10"].notna()
pm25_gt_pm10 = both & (daily["PM2.5"] > daily["PM10"])
invalid_pair = both & (daily["PM2.5"] > 1.5 * daily["PM10"])
local_median = daily.groupby("City")["PM2.5"].transform(
    lambda s: s.rolling(7, center=True, min_periods=3).median())
spike = (daily["PM2.5"] > 300) & (daily["PM2.5"] > 5 * local_median)
pm10_cap = daily["PM10"] >= 1000

invalid = invalid_pair | spike
daily.loc[invalid, ["PM2.5", "PM10"]] = np.nan
daily.loc[pm10_cap, "PM10"] = np.nan
daily_clean = daily.copy()

cleaning_log = pd.DataFrame([
    ("Raw records", len(raw)),
    ("Records after excluding cities", len(df)),
    ("Records added by daily re-indexing", len(daily) - len(df)),
    ("PM2.5 above PM10 (all cases)", int(pm25_gt_pm10.sum())),
    ("PM2.5 above 1.5 x PM10: PM2.5 and PM10 set to missing", int(invalid_pair.sum())),
    ("Isolated PM2.5 spike (above 300 and 5 x local median): PM2.5 and PM10 set to missing", int(spike.sum())),
    ("PM10 at sensor cap (1000 or above): PM10 set to missing", int(pm10_cap.sum())),
    ("PM2.5 above PM10 with ratio up to 1.5: retained", int((pm25_gt_pm10 & ~invalid_pair).sum())),
    ("Records with PM2.5 invalidated", int(invalid.sum())),
], columns=["Step", "Count"])
display(cleaning_log)

In [ ]:
daily["PM2.5_obs"] = daily["PM2.5"]

def fill_short_gaps(s, limit=MAX_FILL):
    isn = s.isna()
    stretch = (isn != isn.shift()).cumsum()
    length = isn.groupby(stretch).transform("size")
    fillable = isn & (length <= limit)
    return s.where(~fillable, s.ffill())

for col in POLLUTANTS:
    daily[col] = daily.groupby("City")[col].transform(fill_short_gaps)
daily["PM10_missing"] = daily["PM10"].isna().astype(np.float32)

after_fill = pd.DataFrame({
    "Missing before filling (%)": daily_clean[POLLUTANTS].isna().mean() * 100,
    "Missing after filling (%)": daily[POLLUTANTS].isna().mean() * 100}).round(1)
display(after_fill)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
for ax, frame, title in [(axes[0], daily_raw, "Before cleaning"), (axes[1], daily_clean, "After cleaning")]:
    ax.scatter(frame["PM10"], frame["PM2.5"], s=3, alpha=0.15, color="#4c78a8")
    ax.plot([0, 1000], [0, 1000], color="#d62728", lw=1, label="PM2.5 = PM10")
    ax.set_xlim(0, 1050); ax.set_ylim(0, 1050)
    ax.set_xlabel("PM10 (µg/m³)"); ax.set_ylabel("PM2.5 (µg/m³)")
    ax.set_title(f"PM2.5 versus PM10: {title}")
    ax.legend(loc="upper left")
steps = cleaning_log.iloc[3:8].copy()
steps["Label"] = ["PM2.5 > PM10", "PM2.5 > 1.5 x PM10", "Isolated spike", "PM10 sensor cap", "Ratio <= 1.5 (kept)"]
axes[2].barh(steps["Label"], steps["Count"], color="#f58518")
axes[2].set_title("Records flagged by cleaning rule")
axes[2].set_xlabel("Records")
axes[2].invert_yaxis()
finish(fig, "02_cleaning_impact")

In [ ]:
daily["row"] = daily.groupby("City").cumcount()
daily["n"] = daily.groupby("City")["row"].transform("size")
train_end = (daily["n"] * TRAIN_FRAC).astype(int)
val_end = (daily["n"] * (TRAIN_FRAC + VAL_FRAC)).astype(int)
daily["period"] = np.select([daily["row"] < train_end, daily["row"] < val_end], [0, 1], 2)
daily["Split"] = daily["period"].map({0: "train", 1: "validation", 2: "test"})

train_rows = daily["period"] == 0
train_median = daily.loc[train_rows, POLLUTANTS].median()
feat = np.log1p(daily[POLLUTANTS].fillna(train_median).clip(lower=0))
feat_mean = feat[train_rows].mean()
feat_std = feat[train_rows].std()
scaled = (feat - feat_mean) / feat_std

month = daily["Date"].dt.month
scaled["PM10_missing"] = daily["PM10_missing"]
scaled["month_sin"] = np.sin(2 * np.pi * month / 12)
scaled["month_cos"] = np.cos(2 * np.pi * month / 12)
CHANNELS = list(scaled.columns)

cities = sorted(daily["City"].unique())
city_to_id = {c: i for i, c in enumerate(cities)}
N_CITIES = len(cities)

X_list, y_list, base_list, pm_win_list = [], [], [], []
cid_list, date_list, split_list = [], [], []
removed_boundary = 0

for city, g in daily.groupby("City"):
    vals = scaled.loc[g.index, CHANNELS].to_numpy(np.float32)
    obs = g["PM2.5_obs"].to_numpy()
    filled = g["PM2.5"].to_numpy()
    period = g["period"].to_numpy()
    dates = g["Date"].to_numpy()
    for i in range(WINDOW - 1, len(g) - HMAX):
        targets = [obs[i + h] for h in HORIZONS]
        if np.isnan(obs[i]) or np.isnan(targets).any():
            continue
        if period[i + 1] != period[i + HMAX]:
            removed_boundary += 1
            continue
        X_list.append(vals[i - WINDOW + 1:i + 1])
        y_list.append(targets)
        base_list.append(obs[i])
        pm_win_list.append(filled[i - WINDOW + 1:i + 1])
        cid_list.append(city_to_id[city])
        date_list.append(dates[i])
        split_list.append(period[i + HMAX])

X = np.stack(X_list).astype(np.float32)
y_abs = np.array(y_list, dtype=np.float32)
base = np.array(base_list, dtype=np.float32)
pm_win = np.stack(pm_win_list).astype(np.float32)
city_id = np.array(cid_list)
win_date = np.array(date_list)
split = np.array(split_list)

y_res = y_abs - base[:, None]
tr, va, te = split == 0, split == 1, split == 2
y_scale = np.maximum(y_res[tr].std(axis=0), 1e-6).astype(np.float32)

print(f"Input tensor: {X.shape} (samples, days, channels) | Targets: {y_abs.shape}")
print(f"Channels: {CHANNELS}")
print(f"Windows: training {tr.sum():,} | validation {va.sum():,} | test {te.sum():,}")
print(f"Windows removed at split boundaries: {removed_boundary}")
print(f"Target scale (training standard deviation of change, µg/m³): {np.round(y_scale, 1)}")

clean_export = daily[["City", "Date"] + POLLUTANTS + ["PM2.5_obs", "PM10_missing", "Split"]].rename(
    columns={"PM2.5_obs": "PM2.5_measured"})
clean_export.to_csv(os.path.join(OUT_DIR, "data", "clean", "clean_daily.csv"), index=False)
np.savez_compressed(os.path.join(OUT_DIR, "data", "clean", "pm25_windows.npz"), X=X, y_abs=y_abs, base=base,
                    pm_win=pm_win, city_id=city_id, split=split, win_date=win_date.astype("datetime64[D]").astype(str),
                    y_scale=y_scale, cities=np.array(cities), channels=np.array(CHANNELS))